# Module 4: Hierarchical Models for Agricultural Commodities

## Learning Objectives
By completing this notebook, you will:
1. Build hierarchical models for the grain complex (corn, wheat, soybeans)
2. Implement crop rotation and land competition constraints
3. Model regional production patterns (US vs. South America)
4. Incorporate stocks-to-use ratios in nonlinear price relationships
5. Ensure forecast consistency across related commodities

## Prerequisites
- Partial pooling concepts from previous notebook
- PyMC model building
- Understanding of agricultural markets and crop rotation

## Estimated Time: 90 minutes

---

## Setup & Imports

We'll use PyMC for hierarchical modeling, ArviZ for diagnostics, and standard scientific Python libraries for data manipulation and visualization.

In [ ]:
# Purpose: Import all required libraries
# Key Concept: Setting up reproducible Bayesian workflow

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az
from scipy import stats

# Set random seeds for reproducibility
np.random.seed(42)
az.style.use('arviz-darkgrid')

# Configure plotting
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")

## Conceptual Introduction

### Why Hierarchical Models for Agriculture?

**Key Insight:** Corn doesn't exist in isolation. Agricultural commodities form interconnected networks through:

1. **Crop Rotation & Land Competition** - Farmers choose between corn and soybeans based on relative profitability
2. **Regional Production Patterns** - Weather shocks affect multiple crops in the same region
3. **Stocks-to-Use Relationships** - Low inventory levels drive price volatility across related crops
4. **Seasonal Patterns** - Harvest timing creates predictable price movements

**Real-world application:** When forecasting corn prices, a hierarchical model prevents logical inconsistencies like predicting prices that would lead farmers to plant 100% corn (economically impossible).

### The Agricultural Pyramid

```
Global Ag Factor (food demand, energy prices, USD)
         │
         ├── Grains Factor
         │        ├── Corn (feed, ethanol)
         │        ├── Wheat (food)
         │        └── Rice (food)
         │
         └── Oilseeds Factor
                  ├── Soybeans (feed, oil)
                  ├── Canola/Rapeseed
                  └── Palm Oil
```

## Part 1: Generate Realistic Agricultural Price Data

Before building models, we'll generate synthetic but realistic price data that exhibits the relationships we expect in agricultural markets.

In [ ]:
# Purpose: Generate synthetic grain complex data with realistic relationships
# Key Concept: Shared global factor + crop-specific dynamics + seasonality

# Time period: 10 years of monthly data
n_months = 120
n_crops = 3
crop_names = ['Corn', 'Wheat', 'Soybeans']

# True global agricultural factor (represents food demand, energy prices, USD strength)
# This is a random walk with small drift
global_ag_factor = np.cumsum(np.random.normal(0.02, 0.5, n_months)) + 4.5

# Crop-specific parameters
# Corn is the reference (intercept = 0)
# Wheat trades at premium (intercept = 1.5)
# Soybeans trade at significant premium (intercept = 5.0)
crop_intercepts = np.array([0.0, 1.5, 5.0])

# Loading on global factor (how much each crop responds to global changes)
crop_loadings = np.array([1.0, 0.85, 0.90])

# Crop-specific noise (idiosyncratic shocks)
crop_noise = np.array([0.3, 0.4, 0.5])

# Generate base prices
prices = np.zeros((n_months, n_crops))
for c in range(n_crops):
    prices[:, c] = (
        crop_intercepts[c] + 
        crop_loadings[c] * global_ag_factor + 
        np.random.normal(0, crop_noise[c], n_months)
    )

# Add harvest seasonality (Northern Hemisphere: Sep-Nov)
# Prices typically drop during harvest as supply floods market
months_idx = np.arange(n_months) % 12
harvest_effect = np.zeros(n_months)
harvest_effect[months_idx == 9] = -0.5   # September
harvest_effect[months_idx == 10] = -0.8  # October (peak harvest)
harvest_effect[months_idx == 11] = -0.5  # November

prices[:, 0] += harvest_effect           # Corn harvest effect
prices[:, 1] += harvest_effect * 0.6     # Wheat (smaller effect, different regions)

# Create dates
dates = pd.date_range('2014-01-01', periods=n_months, freq='MS')

# Create DataFrame
price_df = pd.DataFrame(
    prices,
    columns=crop_names,
    index=dates
)

print("Generated price data:")
print(price_df.describe())
print(f"\nCorn-Soy price ratio (mean): {(prices[:, 0] / prices[:, 2]).mean():.3f}")
print(f"Corn-Soy price ratio (range): [{(prices[:, 0] / prices[:, 2]).min():.3f}, {(prices[:, 0] / prices[:, 2]).max():.3f}]")

In [ ]:
# Purpose: Visualize the grain complex price dynamics
# Key Concept: Identify common trends and crop-specific patterns

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: All prices on same chart
ax = axes[0, 0]
for i, crop in enumerate(crop_names):
    ax.plot(dates, prices[:, i], label=crop, linewidth=2)
ax.set_xlabel('Date')
ax.set_ylabel('Price ($/bushel)')
ax.set_title('Grain Complex Prices', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Corn-Soybean price ratio (key for planting decisions)
ax = axes[0, 1]
ratio = prices[:, 0] / prices[:, 2]
ax.plot(dates, ratio, linewidth=2, color='darkgreen')
ax.axhline(y=ratio.mean(), color='red', linestyle='--', label=f'Mean: {ratio.mean():.3f}')
ax.fill_between(dates, 0.35, 0.50, alpha=0.2, color='gray', label='Typical Range')
ax.set_xlabel('Date')
ax.set_ylabel('Ratio')
ax.set_title('Corn/Soybean Price Ratio', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Seasonal patterns (average by month)
ax = axes[1, 0]
seasonal_avg = price_df.groupby(price_df.index.month).mean()
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
for i, crop in enumerate(crop_names):
    normalized = (seasonal_avg[crop] - seasonal_avg[crop].mean()) / seasonal_avg[crop].std()
    ax.plot(months, normalized, marker='o', label=crop, linewidth=2)
ax.set_xlabel('Month')
ax.set_ylabel('Normalized Price')
ax.set_title('Seasonal Price Patterns (Normalized)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axvspan(8, 11, alpha=0.2, color='orange', label='Harvest')

# Plot 4: Price correlation matrix
ax = axes[1, 1]
corr_matrix = np.corrcoef(prices.T)
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(n_crops))
ax.set_yticks(range(n_crops))
ax.set_xticklabels(crop_names)
ax.set_yticklabels(crop_names)
ax.set_title('Price Correlations', fontsize=14, fontweight='bold')
for i in range(n_crops):
    for j in range(n_crops):
        ax.text(j, i, f'{corr_matrix[i, j]:.2f}', 
               ha='center', va='center', color='black', fontsize=12)
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

print("Key observations:")
print(f"1. Prices are strongly correlated (all > 0.7)")
print(f"2. Harvest months (Sep-Nov) show price declines")
print(f"3. Corn-Soy ratio stays within economic bounds")

## Part 2: Basic Grain Complex Hierarchy

### Mathematical Formulation

**Level 1: Global Agricultural Market Factor**
$$\mu_{\text{global}} \sim \mathcal{N}(5, 2)$$
$$\sigma_{\text{global}} \sim \text{HalfNormal}(1)$$

**Level 2: Global Factor Evolution (Random Walk)**
$$f_t \sim \mathcal{N}(f_{t-1} + \text{drift}, \sigma_{\text{global}})$$

**Level 3: Crop-Specific Parameters (Partial Pooling)**
$$\alpha_c \sim \mathcal{N}(0, 3), \quad c \in \{\text{Corn, Wheat, Soybeans}\}$$
$$\beta_c \sim \mathcal{N}(1, 0.3)$$
$$\sigma_c \sim \text{HalfNormal}(\sigma_{\text{global}})$$

**Level 4: Seasonal Effects**
$$s_{c,m} \sim \mathcal{N}(0, 0.5), \quad m \in \{1, ..., 12\}$$

**Observations:**
$$y_{c,t} \sim \mathcal{N}(\alpha_c + \beta_c f_t + s_{c,m(t)}, \sigma_c)$$

In [ ]:
# Purpose: Build hierarchical model for grain complex
# Key Concept: Partial pooling of information across crops

with pm.Model() as grain_hierarchy:
    # Level 1: Hyperpriors for global agricultural market
    mu_global = pm.Normal('mu_global', mu=5, sigma=2)
    sigma_global = pm.HalfNormal('sigma_global', sigma=1)
    
    # Level 2: Crop-level priors (partial pooling)
    # These allow each crop to have its own parameters while sharing information
    crop_intercept = pm.Normal(
        'crop_intercept',
        mu=0,
        sigma=3,
        shape=n_crops
    )
    
    crop_loading = pm.Normal(
        'crop_loading',
        mu=1,
        sigma=0.3,
        shape=n_crops
    )
    
    crop_sigma = pm.HalfNormal(
        'crop_sigma',
        sigma=sigma_global,
        shape=n_crops
    )
    
    # Level 3: Global agricultural factor (random walk with drift)
    factor_drift = pm.Normal('factor_drift', mu=0, sigma=0.1)
    factor_innov = pm.Normal('factor_innov', mu=0, sigma=1, shape=n_months-1)
    
    # Construct the factor as cumulative sum
    factor = pm.Deterministic(
        'factor',
        pm.math.concatenate([
            [mu_global],
            mu_global + pm.math.cumsum(factor_drift + sigma_global * factor_innov)
        ])
    )
    
    # Level 4: Seasonal effects (one parameter per crop per month)
    seasonal_effect = pm.Normal(
        'seasonal_effect',
        mu=0,
        sigma=0.5,
        shape=(n_crops, 12)
    )
    
    # Likelihood: Observe prices for each crop
    for c in range(n_crops):
        # Get seasonal component for each time point
        seasonal = seasonal_effect[c, months_idx]
        
        # Expected price = intercept + loading * factor + seasonal
        pm.Normal(
            f'price_{crop_names[c]}',
            mu=crop_intercept[c] + crop_loading[c] * factor + seasonal,
            sigma=crop_sigma[c],
            observed=prices[:, c]
        )

print("Model structure:")
print(f"- Global parameters: 2 (mu_global, sigma_global)")
print(f"- Crop parameters: {3 * n_crops} (intercept, loading, sigma for each)")
print(f"- Factor parameters: {n_months} (time-varying)")
print(f"- Seasonal parameters: {n_crops * 12} (12 months × 3 crops)")
print(f"\nTotal parameters: {2 + 3*n_crops + n_months + n_crops*12}")

### Sample from the Posterior

We'll use NUTS (No-U-Turn Sampler) with high target acceptance rate for stable convergence.

In [ ]:
# Purpose: Sample from posterior distribution
# Key Concept: NUTS efficiently explores high-dimensional parameter space

with grain_hierarchy:
    trace_grain = pm.sample(
        1000,
        tune=2000,
        target_accept=0.9,
        return_inferencedata=True,
        random_seed=42
    )

print("\nSampling diagnostics:")
print(az.summary(
    trace_grain,
    var_names=['crop_intercept', 'crop_loading', 'crop_sigma', 'mu_global', 'sigma_global'],
    round_to=3
))

In [ ]:
# Purpose: Visualize crop-level parameter estimates
# Key Concept: Check if model recovered true parameters

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# True values
true_intercepts = crop_intercepts
true_loadings = crop_loadings
true_sigmas = crop_noise

# Plot intercepts
ax = axes[0]
az.plot_forest(
    trace_grain,
    var_names=['crop_intercept'],
    combined=True,
    ax=ax
)
for i, true_val in enumerate(true_intercepts):
    ax.axvline(true_val, color='red', linestyle='--', alpha=0.7)
ax.set_yticklabels(crop_names)
ax.set_title('Crop Intercepts\n(Red = True Values)', fontsize=12, fontweight='bold')
ax.set_xlabel('Price Level ($/bushel)')

# Plot loadings
ax = axes[1]
az.plot_forest(
    trace_grain,
    var_names=['crop_loading'],
    combined=True,
    ax=ax
)
for i, true_val in enumerate(true_loadings):
    ax.axvline(true_val, color='red', linestyle='--', alpha=0.7)
ax.set_yticklabels(crop_names)
ax.set_title('Factor Loadings\n(Sensitivity to Global Factor)', fontsize=12, fontweight='bold')
ax.set_xlabel('Loading')

# Plot sigmas
ax = axes[2]
az.plot_forest(
    trace_grain,
    var_names=['crop_sigma'],
    combined=True,
    ax=ax
)
for i, true_val in enumerate(true_sigmas):
    ax.axvline(true_val, color='red', linestyle='--', alpha=0.7)
ax.set_yticklabels(crop_names)
ax.set_title('Idiosyncratic Volatility', fontsize=12, fontweight='bold')
ax.set_xlabel('Standard Deviation ($/bushel)')

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Intercepts capture base price levels (Soybeans highest, Corn lowest)")
print("- Loadings show sensitivity to global factor (Corn = 1.0 reference)")
print("- Sigmas quantify crop-specific volatility")

In [ ]:
# Purpose: Examine seasonal patterns recovered by the model
# Key Concept: Harvest months should show negative effects

# Extract posterior mean of seasonal effects
seasonal_posterior = trace_grain.posterior['seasonal_effect'].mean(dim=['chain', 'draw']).values

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
months_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

for c, crop in enumerate(crop_names):
    ax = axes[c]
    
    # Plot posterior seasonal pattern
    ax.bar(range(12), seasonal_posterior[c, :], alpha=0.7, color='steelblue')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
    
    # Highlight harvest months
    if c < 2:  # Corn and Wheat have NH harvest
        ax.axvspan(8, 11, alpha=0.2, color='orange')
    
    ax.set_xticks(range(12))
    ax.set_xticklabels(months_labels, rotation=45)
    ax.set_ylabel('Price Effect ($/bushel)')
    ax.set_title(f'{crop} Seasonal Pattern', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nHarvest months (Sep, Oct, Nov) effects:")
for c, crop in enumerate(crop_names):
    sep_effect = seasonal_posterior[c, 9]
    oct_effect = seasonal_posterior[c, 10]
    nov_effect = seasonal_posterior[c, 11]
    print(f"{crop}: Sep={sep_effect:.3f}, Oct={oct_effect:.3f}, Nov={nov_effect:.3f}")
    
print("\nExpected: Negative values during harvest months (supply pressure)")

## Exercise 4.1: Interpret Hierarchical Shrinkage

**Task:** Compare the posterior estimates of crop loadings to what you would get from independent regressions (no hierarchical structure).

**Expected Output:** The hierarchical model should "shrink" extreme estimates toward the group mean (loading ≈ 1.0).

**Hints:**
<details>
<summary>Hint 1</summary>
For independent estimates, regress each crop's prices on the estimated global factor without hierarchical priors.
</details>

<details>
<summary>Hint 2</summary>
Use the posterior mean of the factor from the hierarchical model as your independent variable.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
# 1. Extract posterior mean of global factor
# 2. For each crop, run OLS regression: price ~ factor
# 3. Compare OLS coefficients to hierarchical posterior means
# 4. Plot comparison

from scipy.stats import linregress

# Step 1: Extract factor
factor_mean = trace_grain.posterior['factor'].mean(dim=['chain', 'draw']).values

# Step 2: OLS for each crop
ols_loadings = []
for c in range(n_crops):
    # YOUR CODE: Perform linear regression
    slope, intercept, r_value, p_value, std_err = None, None, None, None, None
    ols_loadings.append(slope)

# Step 3: Extract hierarchical estimates
hierarchical_loadings = trace_grain.posterior['crop_loading'].mean(dim=['chain', 'draw']).values

# Step 4: Plot comparison
# YOUR CODE: Create comparison plot

your_answer = None  # Replace with your comparison insights

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_4_1():
    assert your_answer is not None, "Don't forget to implement your solution!"
    
    # Check that OLS was performed
    assert len(ols_loadings) == n_crops, "Need OLS estimates for all crops"
    assert all(isinstance(x, (int, float)) for x in ols_loadings), "OLS loadings should be numeric"
    
    # Hierarchical estimates should shrink toward mean (loading ≈ 1.0)
    hierarchical_spread = np.std(hierarchical_loadings)
    ols_spread = np.std(ols_loadings)
    
    assert hierarchical_spread < ols_spread, \
        f"Hierarchical model should shrink estimates (reduce spread). Got hierarchical={hierarchical_spread:.3f}, OLS={ols_spread:.3f}"
    
    print("Exercise 4.1 passed!")
    print(f"Hierarchical shrinkage achieved: {(1 - hierarchical_spread/ols_spread)*100:.1f}% reduction in spread")

test_exercise_4_1()

## Part 3: Corn-Soybean Land Competition Model

### Economic Background

Farmers allocate land between corn and soybeans based on expected profitability. The **corn-soybean price ratio** drives planting decisions:

- **High ratio (> 0.50)**: Plant more corn
- **Low ratio (< 0.35)**: Plant more soybeans
- **Normal range**: [0.35, 0.50]

A good forecasting model should respect this constraint. If we predict a ratio of 0.60, we're implicitly predicting farmers will plant 90% corn, which is economically implausible.

### Model Formulation with Economic Constraints

We'll implement a **soft constraint** using `pm.Potential` to penalize forecasts that violate economic bounds.

In [ ]:
# Purpose: Model corn-soy competition with ratio constraints
# Key Concept: Economic constraints prevent implausible forecasts

# Use 5 years of data for this example
n_years = 60
corn_data = prices[:n_years, 0]
soy_data = prices[:n_years, 2]

with pm.Model() as corn_soy_model:
    # Base price levels
    corn_base = pm.Normal('corn_base', mu=4.5, sigma=0.5)
    soy_base = pm.Normal('soy_base', mu=11.0, sigma=1.0)
    
    # Price dynamics (random walks)
    corn_drift = pm.Normal('corn_drift', mu=0, sigma=0.05)
    soy_drift = pm.Normal('soy_drift', mu=0, sigma=0.05)
    
    corn_innov = pm.GaussianRandomWalk(
        'corn_innov',
        mu=corn_drift,
        sigma=0.3,
        shape=n_years
    )
    
    soy_innov = pm.GaussianRandomWalk(
        'soy_innov',
        mu=soy_drift,
        sigma=0.5,
        shape=n_years
    )
    
    # Prices
    corn_price = pm.Deterministic('corn_price', corn_base + corn_innov)
    soy_price = pm.Deterministic('soy_price', soy_base + soy_innov)
    
    # Corn-Soy ratio
    ratio = pm.Deterministic('ratio', corn_price / soy_price)
    
    # Economic constraint: Ratio should stay in [0.30, 0.55]
    # Use potential to add log-probability penalty for violations
    ratio_penalty_lower = pm.math.switch(
        ratio < 0.30,
        -50 * (0.30 - ratio)**2,  # Strong penalty below 0.30
        0
    )
    
    ratio_penalty_upper = pm.math.switch(
        ratio > 0.55,
        -50 * (ratio - 0.55)**2,  # Strong penalty above 0.55
        0
    )
    
    pm.Potential('ratio_constraint', ratio_penalty_lower + ratio_penalty_upper)
    
    # Observation noise
    corn_sigma = pm.HalfNormal('corn_sigma', sigma=0.3)
    soy_sigma = pm.HalfNormal('soy_sigma', sigma=0.5)
    
    # Likelihood
    pm.Normal('corn_obs', mu=corn_price, sigma=corn_sigma, observed=corn_data)
    pm.Normal('soy_obs', mu=soy_price, sigma=soy_sigma, observed=soy_data)
    
    # Sample
    trace_competition = pm.sample(
        1000,
        tune=1500,
        target_accept=0.9,
        return_inferencedata=True,
        random_seed=42
    )

print("\nCorn-Soy model summary:")
print(az.summary(
    trace_competition,
    var_names=['corn_base', 'soy_base', 'corn_sigma', 'soy_sigma'],
    round_to=3
))

In [ ]:
# Purpose: Verify that ratio constraint is respected
# Key Concept: Posterior predictive should stay within economic bounds

# Extract ratio from posterior
ratio_posterior = trace_competition.posterior['ratio'].values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Ratio over time with credible intervals
ax = axes[0]
ratio_mean = ratio_posterior.mean(axis=(0, 1))
ratio_lower = np.percentile(ratio_posterior, 5, axis=(0, 1))
ratio_upper = np.percentile(ratio_posterior, 95, axis=(0, 1))

time_idx = np.arange(n_years)
ax.plot(time_idx, ratio_mean, linewidth=2, label='Posterior Mean')
ax.fill_between(time_idx, ratio_lower, ratio_upper, alpha=0.3, label='90% CI')

# Show economic bounds
ax.axhline(0.30, color='red', linestyle='--', label='Lower Bound')
ax.axhline(0.55, color='red', linestyle='--', label='Upper Bound')
ax.fill_between(time_idx, 0.35, 0.50, alpha=0.1, color='green', label='Typical Range')

ax.set_xlabel('Month')
ax.set_ylabel('Corn/Soy Ratio')
ax.set_title('Corn-Soy Price Ratio Over Time', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Distribution of ratio values
ax = axes[1]
ratio_flat = ratio_posterior.flatten()
ax.hist(ratio_flat, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='black')
ax.axvline(0.30, color='red', linestyle='--', linewidth=2, label='Constraint Bounds')
ax.axvline(0.55, color='red', linestyle='--', linewidth=2)
ax.axvspan(0.35, 0.50, alpha=0.1, color='green', label='Typical Range')
ax.set_xlabel('Corn/Soy Ratio')
ax.set_ylabel('Density')
ax.set_title('Posterior Distribution of Ratio', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Calculate violations
violations_lower = (ratio_flat < 0.30).sum()
violations_upper = (ratio_flat > 0.55).sum()
total_samples = len(ratio_flat)

print(f"\nConstraint violations:")
print(f"Below 0.30: {violations_lower}/{total_samples} ({100*violations_lower/total_samples:.2f}%)")
print(f"Above 0.55: {violations_upper}/{total_samples} ({100*violations_upper/total_samples:.2f}%)")
print(f"\nInterpretation: Constraint successfully limits extreme ratio values")

## Exercise 4.2: Incorporate Stocks-to-Use Ratio

**Task:** Extend the corn-soy model to include the stocks-to-use ratio as a covariate. Price should be a **nonlinear function** of stocks:

$$P_t = \alpha + \frac{\beta}{S_t} + \epsilon_t$$

Where $S_t$ is the stocks-to-use ratio (0.10 to 0.35).

**Expected Output:** 
- When stocks are low (< 0.15), prices should be high and volatile
- When stocks are comfortable (> 0.25), prices should be stable

**Hints:**
<details>
<summary>Hint 1</summary>
Generate synthetic stocks-to-use data: np.random.uniform(0.10, 0.35, n_years)
</details>

<details>
<summary>Hint 2</summary>
Use pm.Deterministic to compute expected_price = alpha + beta / stocks_ratio
</details>

<details>
<summary>Hint 3</summary>
Make volatility depend on stocks: sigma_t = sigma_base + sigma_stocks / stocks_ratio
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
# 1. Generate stocks-to-use data
# 2. Build PyMC model with nonlinear price-stocks relationship
# 3. Sample from posterior
# 4. Plot: price vs stocks-to-use with posterior uncertainty

n_obs = 60
stocks_to_use_ratio = np.random.uniform(0.10, 0.35, n_obs)

with pm.Model() as stocks_model:
    # YOUR CODE: Define model
    # Parameters: alpha (base price), beta (scarcity premium)
    # Expected price = alpha + beta / stocks_ratio
    # Sigma also depends on stocks
    
    pass  # Remove this and implement

# YOUR CODE: Sample and visualize

your_answer = None  # Replace with your trace

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_4_2():
    assert your_answer is not None, "Don't forget to implement your solution!"
    assert isinstance(your_answer, az.InferenceData), "Should return InferenceData object"
    
    # Check that model includes alpha and beta
    assert 'alpha' in your_answer.posterior, "Model should have 'alpha' parameter"
    assert 'beta' in your_answer.posterior, "Model should have 'beta' parameter (scarcity premium)"
    
    # Beta should be positive (low stocks → high prices)
    beta_samples = your_answer.posterior['beta'].values.flatten()
    assert np.mean(beta_samples) > 0, "Beta should be positive (inverse relationship)"
    
    print("Exercise 4.2 passed!")
    print(f"Scarcity premium (beta): {np.mean(beta_samples):.2f} ± {np.std(beta_samples):.2f}")

test_exercise_4_2()

## Part 4: Regional Hierarchy - US vs. South America

### Key Insight

Soybeans are grown in opposite hemispheres:
- **US harvest**: September-November
- **Brazil harvest**: February-April
- **Argentina harvest**: April-June

This creates **seasonal arbitrage opportunities** and means global supply is staggered throughout the year.

A regional hierarchy captures:
1. Global soybean demand (China imports)
2. Regional basis (transportation costs, local policies)
3. Opposite-hemisphere seasonality

In [ ]:
# Purpose: Generate regional soybean price data
# Key Concept: Shared global factor + region-specific basis + opposite seasons

n_obs = 100
regions = ['US', 'Brazil', 'Argentina']
n_regions = len(regions)

# Global soybean factor
global_soy = np.cumsum(np.random.normal(0.02, 0.4, n_obs)) + 10.0

# Regional basis (transport costs, export taxes)
basis = {
    'US': 0.0,        # Reference price
    'Brazil': -0.3,   # Discount due to export taxes
    'Argentina': -0.5 # Larger discount
}

# Generate prices with seasonal effects
months_idx = np.arange(n_obs) % 12
soy_prices = np.zeros((n_obs, n_regions))

for i, region in enumerate(regions):
    # Base price = global + basis
    base_price = global_soy + basis[region]
    
    # Add harvest effect (prices drop during harvest)
    harvest_effect = np.zeros(n_obs)
    if region == 'US':
        harvest_months = [9, 10, 11]  # Sep-Nov
    elif region == 'Brazil':
        harvest_months = [2, 3, 4]    # Feb-Apr
    else:  # Argentina
        harvest_months = [4, 5, 6]    # Apr-Jun
    
    for month in harvest_months:
        harvest_effect[months_idx == month] = -0.5
    
    soy_prices[:, i] = base_price + harvest_effect + np.random.normal(0, 0.3, n_obs)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Time series
ax = axes[0]
for i, region in enumerate(regions):
    ax.plot(soy_prices[:, i], label=region, linewidth=2)
ax.set_xlabel('Month')
ax.set_ylabel('Price ($/bushel)')
ax.set_title('Regional Soybean Prices', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Seasonal patterns
ax = axes[1]
for i, region in enumerate(regions):
    seasonal_avg = np.array([soy_prices[months_idx == m, i].mean() for m in range(12)])
    seasonal_norm = seasonal_avg - seasonal_avg.mean()
    ax.plot(months_labels, seasonal_norm, marker='o', label=region, linewidth=2)
ax.set_xlabel('Month')
ax.set_ylabel('Deviation from Mean ($/bushel)')
ax.set_title('Seasonal Price Patterns by Region', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.show()

print("Regional price differences:")
print(f"US mean: ${soy_prices[:, 0].mean():.2f}")
print(f"Brazil mean: ${soy_prices[:, 1].mean():.2f} (discount: ${soy_prices[:, 0].mean() - soy_prices[:, 1].mean():.2f})")
print(f"Argentina mean: ${soy_prices[:, 2].mean():.2f} (discount: ${soy_prices[:, 0].mean() - soy_prices[:, 2].mean():.2f})")

## Exercise 4.3: Build Regional Hierarchy

**Task:** Implement a hierarchical model for regional soybean prices with:

1. **Level 1**: Global soybean factor (shared across regions)
2. **Level 2**: Regional basis (transportation costs, policies)
3. **Level 3**: Region-specific seasonal effects

**Expected Output:** 
- Global factor captures common trends
- Basis captures persistent price differentials
- Seasonal effects show opposite-hemisphere harvest timing

**Hints:**
<details>
<summary>Hint 1</summary>
Use pm.GaussianRandomWalk for global factor evolution
</details>

<details>
<summary>Hint 2</summary>
Regional basis: pm.Normal('region_basis', mu=0, sigma=1, shape=n_regions)
</details>

<details>
<summary>Hint 3</summary>
Seasonal effects: shape=(n_regions, 12) for each region-month combination
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
# Implement regional soybean hierarchy

with pm.Model() as regional_soy_model:
    # YOUR CODE: Build hierarchical model
    # Level 1: Global factor
    # Level 2: Regional basis
    # Level 3: Seasonal effects
    # Likelihood: Observed prices for each region
    
    pass  # Remove this and implement

# YOUR CODE: Sample and analyze

your_answer = None  # Replace with your trace

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_4_3():
    assert your_answer is not None, "Don't forget to implement your solution!"
    assert isinstance(your_answer, az.InferenceData), "Should return InferenceData object"
    
    # Check model structure
    assert 'region_basis' in your_answer.posterior, "Need regional basis parameters"
    assert 'seasonal_effect' in your_answer.posterior or 'seasonal_harvest' in your_answer.posterior, \
        "Need seasonal effects"
    
    # Check that Brazil and Argentina have negative basis
    basis_samples = your_answer.posterior['region_basis'].values
    brazil_basis = basis_samples[:, :, 1].mean()
    argentina_basis = basis_samples[:, :, 2].mean()
    
    assert brazil_basis < 0, "Brazil should have negative basis (discount to US)"
    assert argentina_basis < brazil_basis, "Argentina should have larger discount than Brazil"
    
    print("Exercise 4.3 passed!")
    print(f"Regional basis (relative to US):")
    print(f"  Brazil: ${brazil_basis:.2f}")
    print(f"  Argentina: ${argentina_basis:.2f}")

test_exercise_4_3()

## Summary

### Key Takeaways

1. **Hierarchical structures respect market relationships** - Grain complex models prevent inconsistent forecasts

2. **Economic constraints are essential** - Corn-soy ratio constraints ensure forecasts are economically plausible

3. **Partial pooling improves estimates** - Borrowing strength across crops reduces uncertainty

4. **Regional hierarchies capture geographic patterns** - Opposite-hemisphere harvest timing affects global markets

5. **Nonlinear relationships matter** - Stocks-to-use has convex relationship with prices

### What's Next

**Module 5: Gaussian Processes** - Model complex temporal patterns and smooth price curves without parametric assumptions.

**Module 7: Regime Switching** - Capture structural breaks in agricultural markets (policy changes, technology adoption).

### Additional Resources

1. **Wright, B.D. (2011)**. "The Economics of Grain Price Volatility." *Applied Economic Perspectives and Policy*
2. **Roberts & Schlenker (2013)**. "Identifying Supply and Demand Elasticities of Agricultural Commodities." *Review of Economic Studies*
3. **USDA WASDE Reports** - Monthly supply/demand estimates: [https://www.usda.gov/oce/commodity/wasde/](https://www.usda.gov/oce/commodity/wasde/)

---

*"In agricultural markets, every crop is a neighbor. Hierarchical models respect the community."*